Chapter 3 covers Time Series Analysis. In business analytics and risk modeling, time-series data usually arrives as granular event logs (individual transactions, sensor readings, system events).

The primary goal of Chapter 3 is learning how to transform point-in-time transactions into continuous, structured temporal aggregations (daily, monthly, quarterly) and perform relative period-over-period comparisons inside the database engine.

The 5 Core Takeaways of Chapter 3

1. Temporal Bucketing & Truncation

To group time-series data by time buckets (e.g., converting 2026-08-01 11:58:14 into 2026-08-01 or 2026-08), databases use date truncation functions.

Key Concept: DATE_TRUNC('month', date_column) chops off the trailing precision, snapping every timestamp to the beginning of its interval.

Why it matters: It gives you a clean key to execute GROUP BY operations over uniform temporal windows.

2. Window Functions over Time (OVER (...))

Instead of collapsing rows with a standard GROUP BY, window functions compute metrics across a sliding frame while keeping all individual rows intact.

Syntax Structure: FUNCTION() OVER (PARTITION BY group_col ORDER BY time_col [RANGE/ROWS frame])

Core Application: Calculating moving averages, cumulative sums (running totals), and rolling volatility metrics directly on set partitions.

3. Lag and Lead Operations (Period-over-Period Deltas)

Comparing current metrics to past or future periods (e.g., month-over-month growth $M/M$ or year-over-year $Y/Y$) requires accessing adjacent rows without writing recursive self-joins.

LAG(column, offset): Reaches back $N$ rows in the partition order (e.g., prior month's sales).

LEAD(column, offset): Reaches forward $N$ rows.4. Handling Missing Periods (Date Spines / Saffrons)

If a business makes zero sales on a given day, SQL's GROUP BY will completely drop that date from the query output. This creates discontinuous time series that break downstream quantitative models.

The Solution: Generating a continuous "date spine" (using GENERATE_SERIES() in DuckDB/Postgres) and LEFT JOIN-ing the event table against it, wrapping missing values in COALESCE(sales, 0).5. Cumulative & Year-to-Date (YTD) MetricsTracking cumulative progress requires bounding window frames relative to calendar anchor dates.

YTD Calculation: SUM(sales) OVER (PARTITION BY YEAR(sales_month) ORDER BY sales_month)


In [ ]:
import duckdb

conn = duckdb.connect(database=":memory:")

# Ingest local CSV dataset
conn.execute(
    "CREATE TABLE retail_sales AS SELECT * FROM read_csv_auto('us_retail_sales.csv');"
)

# Chapter 3 Execution: Moving Averages, Prior Period Comparisons, and YoY Deltas
ch3_analysis_df = conn.execute("""
WITH monthly_total AS (
    -- Step 1: Bucket and Aggregate Total Sales by Month
    SELECT
        CAST(sales_month AS DATE) AS sales_month,
        SUM(sales) AS total_sales
    FROM retail_sales
    WHERE kind_of_business = 'Retail and food services sales, total'
    GROUP BY 1
),
time_series_metrics AS (
    -- Step 2: Apply Window Functions for Rolling Windows & Lags
    SELECT
        sales_month,
        total_sales,

        -- Prior Month Sales (M/M)
        LAG(total_sales, 1) OVER (ORDER BY sales_month) AS prev_month_sales,

        -- Prior Year Sales (Y/Y)
        LAG(total_sales, 12) OVER (ORDER BY sales_month) AS prev_year_sales,

        -- 12-Month Moving Average
        AVG(total_sales) OVER (
            ORDER BY sales_month
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS rolling_12mo_avg
    FROM monthly_total
)
-- Step 3: Compute Relative Differences
SELECT
    sales_month,
    total_sales,
    rolling_12mo_avg,

    -- Month-over-Month % Change
    ROUND(((total_sales - prev_month_sales) / prev_month_sales) * 100, 2) AS mom_pct_change,

    -- Year-over-Year % Change
    ROUND(((total_sales - prev_year_sales) / prev_year_sales) * 100, 2) AS yoy_pct_change
FROM time_series_metrics
ORDER BY sales_month DESC
LIMIT 12;
""").df()

ch3_analysis_df

| Concept              | Problem Solved                                                     | Primary SQL Syntax                                                       |
| :------------------- | :----------------------------------------------------------------- | :----------------------------------------------------------------------- |
| **Date Truncation**  | Collapsing irregular timestamps into uniform daily/monthly buckets | `DATE_TRUNC('month', date)`                                              |
| **Moving Averages**  | Smoothing out seasonal noise and high-frequency volatility         | `AVG(val) OVER (ORDER BY date ROWS BETWEEN N PRECEDING AND CURRENT ROW)` |
| **MoM / YoY Deltas** | Measuring period-over-period growth or rate of change              | `LAG(val, 1) OVER (ORDER BY date)` / `LAG(val, 12)`                      |
| **Date Spines**      | Fixing missing dates/gaps in time-series sequences                 | `GENERATE_SERIES(start, end, interval) + LEFT JOIN`                      |


In [13]:
import duckdb

# 1. Connect and ingest local CSV
conn = duckdb.connect(database=":memory:")
conn.execute(
    "CREATE TABLE retail_sales AS SELECT * FROM read_csv_auto('us_retail_sales.csv');"
)

# 2. Read the full .sql file
with open("time series queries.sql", "r", encoding="utf-8") as f:
    raw_sql = f.read()

# 3. Clean and split queries by semicolon
queries = [q.strip() for q in raw_sql.split(";") if q.strip()]

print(f"Successfully loaded {len(queries)} queries from 'time series queries.sql'.")

Successfully loaded 34 queries from 'time series queries.sql'.


In [17]:
# Execute Query #1 (Chapter 3 First Example)
df = conn.execute(queries[0]).df()
df.head(10)

,sales_month,sales
0,1992-01-01,146376
1,1992-02-01,147079
2,1992-03-01,159336
3,1992-04-01,163669
4,1992-05-01,170068
5,1992-06-01,168663
6,1992-07-01,169890
7,1992-08-01,170364
8,1992-09-01,164617
9,1992-10-01,173655


In [ ]:
%%sql
-- Directly test CTEs, moving averages, or cohort dates from Chapter 3
SELECT
    sales_month,
    kind_of_business,
    sales,
    AVG(sales) OVER(
        PARTITION BY kind_of_business
        ORDER BY sales_month
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ) AS rolling_12mo_avg
FROM retail_sales
WHERE kind_of_business = 'Retail and food services sales, total'
LIMIT 12;

Running query in 'duckdb:///:memory:'

sales_month,kind_of_business,sales,rolling_12mo_avg
1992-01-01,"Retail and food services sales, total",146376,146376.0
1992-02-01,"Retail and food services sales, total",147079,146727.5
1992-03-01,"Retail and food services sales, total",159336,150930.33333333334
1992-04-01,"Retail and food services sales, total",163669,154115.0
1992-05-01,"Retail and food services sales, total",170068,157305.6
1992-06-01,"Retail and food services sales, total",168663,159198.5
1992-07-01,"Retail and food services sales, total",169890,160725.85714285713
1992-08-01,"Retail and food services sales, total",170364,161930.625
1992-09-01,"Retail and food services sales, total",164617,162229.11111111112
1992-10-01,"Retail and food services sales, total",173655,163371.7
